# 01 - Bronze: Ingest raw CSVs (schema-per-league design)

One schema per competition, one table per season inside it:

    workspace.bronze_england_premier_league.season_2005_06
    workspace.bronze_england_premier_league.season_2006_07
    ...
    workspace.bronze_argentina.all_seasons

Databricks only supports catalog.schema.table (3 levels) - so the layer
(bronze) and league name are folded together into the schema name
(prefixed bronze_ so schema names never start with a digit, e.g. "2. Bundesliga"
would otherwise become an invalid identifier).

Resumable: skips any table that already exists, safe to re-run if
interrupted.

In [ ]:
import os
import re
import pandas as pd

RAW_PATH = "/Volumes/workspace/default/football_raw"

## Step 1: Discover files

In [ ]:
main_files = []
for root, dirs, files in os.walk(f"{RAW_PATH}/main_leagues"):
    for f in files:
        if f.endswith(".csv"):
            main_files.append(os.path.join(root, f))

extra_files = []
for root, dirs, files in os.walk(f"{RAW_PATH}/extra_leagues"):
    for f in files:
        if f.endswith(".csv"):
            extra_files.append(os.path.join(root, f))

print(f"Main league files: {len(main_files)}")
print(f"Extra league files: {len(extra_files)}")

## Step 2: Helper functions

In [ ]:
def read_csv_robust(path):
    try:
        df = pd.read_csv(path, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="cp1252")
    # Drop ghost "Unnamed: N" columns from trailing commas in source files
    unnamed = [c for c in df.columns if str(c).startswith("Unnamed:")]
    return df.drop(columns=unnamed)

def sanitize(name):
    """Turn a league/country name into a valid schema/table identifier."""
    name = name.lower().strip()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

def parse_main_path(path):
    """.../main_leagues/{country}/{division}/{season}.csv"""
    parts = path.replace("\\", "/").split("/")
    country, division, filename = parts[-3], parts[-2], parts[-1]
    season = filename.replace(".csv", "")
    schema_name = f"bronze_{sanitize(country)}_{sanitize(division)}"
    table_name = f"season_{season.replace('-', '_')}"
    return schema_name, table_name, country, division, season

EXTRA_CODE_TO_COUNTRY = {
    "ARG": "argentina", "AUT": "austria", "BRA": "brazil", "CHN": "china",
    "DNK": "denmark", "FIN": "finland", "IRL": "ireland", "JPN": "japan",
    "MEX": "mexico", "NOR": "norway", "POL": "poland", "ROU": "romania",
    "RUS": "russia", "SWE": "sweden", "SWZ": "switzerland", "USA": "usa",
}

def parse_extra_path(path):
    """.../extra_leagues/{CODE}.csv"""
    code_ = path.replace("\\", "/").split("/")[-1].replace(".csv", "")
    country = EXTRA_CODE_TO_COUNTRY.get(code_, code_)
    schema_name = f"bronze_{sanitize(country)}"
    return schema_name, "matches", code_

## Step 3: Ingest main leagues - one schema per league, one table per season

Each file becomes its OWN table, so `mode("overwrite")` is correct here
(not append - there's nothing to append to, each season is a distinct
table). Skips tables that already exist, so re-running after an
interruption is safe.

In [ ]:
ingested, skipped, failed = 0, 0, []

for f in main_files:
    schema_name, table_name, country, division, season = parse_main_path(f)
    full_table = f"workspace.{schema_name}.{table_name}"

    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{schema_name}")

        if spark.catalog.tableExists(full_table):
            skipped += 1
            continue

        df = read_csv_robust(f)
        df["source_country"] = country
        df["source_division"] = division
        df["source_season"] = season
        df["source_file"] = f
        df = df.astype(str)

        spark_df = spark.createDataFrame(df)
        spark_df.write.format("delta").mode("overwrite").saveAsTable(full_table)

        ingested += 1
        if ingested % 50 == 0:
            print(f"  ...{ingested} tables created")

    except Exception as e:
        failed.append((f, full_table, str(e)))

print(f"\nMain leagues: {ingested} tables created, {skipped} already existed, {len(failed)} failed")
for f, t, err in failed[:10]:
    print(f"  FAILED: {t} <- {f}")
    print(f"    {err[:200]}")

## Step 4: Ingest extra leagues - one schema per league, one table (all seasons already in the file)

In [ ]:
ingested_extra, skipped_extra, failed_extra = 0, 0, []

for f in extra_files:
    schema_name, table_name, code_ = parse_extra_path(f)
    full_table = f"workspace.{schema_name}.{table_name}"

    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.{schema_name}")

        if spark.catalog.tableExists(full_table):
            skipped_extra += 1
            continue

        df = read_csv_robust(f)
        df["source_code"] = code_
        df["source_file"] = f
        df = df.astype(str)

        spark_df = spark.createDataFrame(df)
        spark_df.write.format("delta").mode("overwrite").saveAsTable(full_table)

        ingested_extra += 1

    except Exception as e:
        failed_extra.append((f, full_table, str(e)))

print(f"Extra leagues: {ingested_extra} tables created, {skipped_extra} already existed, {len(failed_extra)} failed")
for f, t, err in failed_extra:
    print(f"  FAILED: {t} <- {f}")
    print(f"    {err[:200]}")

## Step 5: Verify - list all schemas and count tables in each

In [ ]:
schemas = [row.databaseName for row in spark.sql("SHOW SCHEMAS IN workspace").collect()
           if row.databaseName.startswith("bronze_")]

print(f"Total league schemas created: {len(schemas)}\n")

total_tables = 0
for s in sorted(schemas):
    tables = spark.sql(f"SHOW TABLES IN workspace.{s}").collect()
    total_tables += len(tables)
    print(f"  {s}: {len(tables)} table(s)")

print(f"\nTotal tables across all schemas: {total_tables}")